# Table Access Protocol (TAP) queries: epochs
Ref: [TAP](https://dp1.lsst.io/tutorials/notebook/102/notebook-102-1.html)

In [1]:
import numpy as np
from lsst.rsp import get_tap_service
import pandas as pd
import matplotlib.pyplot as plt
import time

service = get_tap_service ('tap')
assert service is not None

fields = pd.read_csv('LSSTComCam_fields.csv', index_col='field')

ddeg = .01
def grabcone(field):
    return "CONTAINS(POINT('ICRS', coord_ra, coord_dec), " +\
           f"CIRCLE('ICRS', {fields.loc[field, 'ra_deg']}, " +\
                          f"{fields.loc[field, 'dec_deg']}, {ddeg})) = 1"

In [2]:
queries = pd.DataFrame(columns=['select', 'from', 'where'])
queries.loc['columns'] = ["column_name, datatype, description, unit",
                          "tap_schema.columns",
                          "table_name = 'dp1.Object'"]
qcols = queries.columns

query = ''
for col in qcols:
    if queries.loc['columns'][col] != '':
        query += f' {col.upper()} {queries.loc['columns'][col]}'
print(query)

job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
if job.phase == 'ERROR':
    job.raise_if_error()
elif job.phase == 'COMPLETED':
    out = job.fetch_result().to_table()

queries.loc['columns', 'out'] = out
print(type(out), len(out))
#       display(out)
print()

 SELECT column_name, datatype, description, unit FROM tap_schema.columns WHERE table_name = 'dp1.Object'
<class 'astropy.table.table.Table'> 1296



In [3]:
ocols = queries.loc['columns', 'out'].to_pandas()
ocols

,column_name,datatype,description,unit
0,coord_dec,double,Fiducial ICRS Declination of centroid used for...,deg
1,coord_decErr,float,Error in fiducial ICRS Declination of centroid,deg
2,coord_ra,double,Fiducial ICRS Right Ascension of centroid used...,deg
3,coord_ra_dec_Cov,float,Covariance between fiducial ICRS Right Ascensi...,deg**2
4,coord_raErr,float,Error in fiducial ICRS Right Ascension of cent...,deg
...,...,...,...,...
1291,z_raErr,float,"Error in right ascension, measured on z-band.",deg
1292,z_sersicFlux,float,z-band flux from the multiband Sersic model fit.,nJy
1293,z_sersicFluxErr,float,Error on the z-band flux from the multiband Se...,nJy
1294,z_sizeExtendedness,float,Moments-based measure of whether an object is ...,


In [22]:
ocols.loc[ocols['column_name'].str.contains('epoch', case=False)]

,column_name,datatype,description,unit
94,g_epoch,double,Mean epoch of the object in the g-band coadd i...,d
300,i_epoch,double,Mean epoch of the object in the i-band coadd i...,d
509,r_epoch,double,Mean epoch of the object in the r-band coadd i...,d
744,u_epoch,double,Mean epoch of the object in the u-band coadd i...,d
954,y_epoch,double,Mean epoch of the object in the y-band coadd i...,d
1161,z_epoch,double,Mean epoch of the object in the z-band coadd i...,d
